# 無人機影像超解析度 · Google Colab

先在 **執行階段 → 變更執行階段類型 → GPU** 選擇 GPU，再連線。執行第 1、2 步後，在第 2 步的下拉選單選擇權重，再依序執行第 3–6 步。

- **單張上傳**：在第 3 步選 `upload`，上傳一張 JPG／JPEG／PNG／TIF／TIFF。
- **Google Drive 圖片資料夾**：在第 3 步選 `drive`、填寫輸入／輸出資料夾。預設只取第一張；確認結果與耗時後，將 `DRIVE_LIMIT` 改成 `0` 才會處理全部。
- **模型權重**：第 2 步掛載 Drive，預設掃描 `/content/drive/MyDrive/models` 第一層，從下拉選單選擇權重；可修改資料夾路徑。請先將權重放入該資料夾，筆記本不自動下載權重。
- 使用本專案既有 Spandrel／分塊推論，放大倍率由選取的模型決定。保留原圖，輸出 RGB PNG、逐張狀態與 `run.log`；每次推論建立新目錄。
- 只讀資料夾第一層；不支援高位深、多頁圖片。同 stem 的不同圖片會由原 CLI 回報衝突。輸出不保留 EXIF／GIS／alpha。

此版本取用公開專案 commit `7a7c6ee8859754d5162bec032133b5bd7d2a103e` 的五個原始碼檔；**本機尚未提交的修改不會自動帶入 Colab**。保留 Colab 預裝的 PyTorch、torchvision、NumPy、Pillow，只安裝同專案版本的 Spandrel 及缺少的傳遞依賴，不使用 WSL 的 CUDA 安裝清單。

Colab GPU 型號、配額及工作階段壽命不固定；沒有 GPU 時本筆記本停止。`/content` 檔案會隨執行環境刪除，單張上傳模式請在結束前下載結果。Drive 模式直接存到指定輸出資料夾。詳見 [Colab 官方 FAQ](https://research.google.com/colaboratory/faq.html)。

**驗證界線（2026-10-02）**：WSL 已通過結構／語法檢查，以及模擬 Drive／選單／推論的權重掃描、切換模型、空資料夾、移除檔案、重掃清除舊選項、CLI 參數與 SHA-256 紀錄檢查。原有上傳／Drive 圖片、部分失敗與 ZIP 流程先前已通過模擬檢查。Google 登入、Drive 掛載、實際選單互動、Colab 套件安裝與 GPU 推論尚未實測，須在實際 Colab 工作階段驗證。先以一張小圖確認，勿直接啟動完整資料集。

## 1. 檢查 GPU 與準備套件

需要 Python 3.12 以上的 Colab 託管 GPU 環境。這一步先實測 CUDA，再安裝 Spandrel；不更換平台預裝的 PyTorch／CUDA。如果現有版本無法匯入或依賴解析失敗，請保留錯誤訊息，不要繼續執行。

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import tempfile
import time
from pathlib import Path
from importlib.metadata import version

if sys.version_info < (3, 12):
    raise RuntimeError("需要 Python 3.12 以上，請選用符合條件的 Colab 執行階段。")
from google.colab import files, drive

WORKSPACE = Path("/content/drone-sr-colab")
WORKSPACE.mkdir(parents=True, exist_ok=True)
GPU_CHECK = """
import torch, torchvision
from PIL import Image
if not torch.cuda.is_available():
    raise SystemExit('沒有可用的 CUDA GPU；請變更 Colab 執行階段類型後重試。')
layer = torch.nn.Conv2d(3, 3, 3).cuda()
layer(torch.zeros(1, 3, 8, 8, device='cuda')).sum().item()
free, total = torch.cuda.mem_get_info()
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM free/total GiB:', round(free / 2**30, 2), '/', round(total / 2**30, 2))
print('torch:', torch.__version__, 'torchvision:', torchvision.__version__, 'CUDA:', torch.version.cuda)
"""
print("Python:", sys.version.split()[0])
print("本機暫存可用磁碟 GiB:", round(shutil.disk_usage(WORKSPACE).free / 2**30, 2))
for line in Path("/proc/meminfo").read_text().splitlines():
    if line.startswith("MemAvailable:"):
        print("可用 RAM GiB:", round(int(line.split()[1]) / 2**20, 2))
subprocess.run([sys.executable, "-c", GPU_CHECK], check=True)

# Constrain the hosted runtime packages so pip cannot replace its CUDA stack.
protected = ("torch", "torchvision", "numpy", "Pillow")
constraints = WORKSPACE / "runtime-constraints.txt"
constraints.write_text("\n".join(f"{name}=={version(name)}" for name in protected) + "\n")
subprocess.run([
    sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
    "--constraint", str(constraints), "spandrel==0.4.2",
], check=True)
subprocess.run([
    sys.executable, "-c",
    "from spandrel import ModelLoader; from torchvision.transforms.functional import pil_to_tensor; "
    "from importlib.metadata import version; "
    "print({name: version(name) for name in ('torch', 'torchvision', 'spandrel', 'Pillow')})",
], check=True)

## 2. 掛載 Drive 並選擇模型權重

先把權重放在 Google Drive「我的雲端硬碟」下的 `models` 資料夾。預設路徑為 `/content/drive/MyDrive/models`；若放在其他位置，修改下方 `DRIVE_MODELS`，再執行此步。

掛載後會把資料夾第一層的 `.pth`、`.pt`、`.ckpt`、`.safetensors` 檔案列成下拉選單，請選擇一顆再執行後續步驟。選單依副檔名列出候選檔案，實際仍須符合 Spandrel 支援的 RGB 超解析度模型。加入、移除權重或改變資料夾後，**重跑此步即可更新清單**；僅切換現有選項時，可直接重跑第 4–6 步。

程式仍從指定 commit 下載 `src/drone_sr/`，權重則直接讀取所選 Drive 檔案。執行時會記錄權重路徑與 SHA-256，供追溯本輪使用的檔案；這不是與官方校驗值比對。

In [ ]:
#@title Drive 模型資料夾與權重選單
DRIVE_MODELS = "/content/drive/MyDrive/models" #@param {type:"string"}

if globals().get("model_dropdown") is not None:
    model_dropdown.close()
model_dropdown = None
MODEL_PATH = None
MODEL_SHA256 = None

import hashlib
from urllib.request import urlopen
import ipywidgets as widgets
from IPython.display import display

drive.mount("/content/drive")
model_directory = Path(DRIVE_MODELS).expanduser()
if not model_directory.is_absolute():
    raise ValueError("DRIVE_MODELS 必須是掛載後的絕對路徑。")
model_directory = model_directory.resolve()
if not model_directory.is_relative_to(Path("/content/drive").resolve()):
    raise ValueError("模型資料夾必須位於 /content/drive/ 下。")
if not model_directory.is_dir():
    raise FileNotFoundError(f"找不到模型資料夾：{model_directory}；請在 Drive 建立資料夾並放入權重後重跑。")
checkpoint_suffixes = {".pth", ".pt", ".ckpt", ".safetensors"}
checkpoints = sorted(
    (p for p in model_directory.iterdir() if p.is_file() and p.suffix.lower() in checkpoint_suffixes),
    key=lambda p: (p.name.casefold(), p.name),
)
if not checkpoints:
    raise ValueError(f"{model_directory} 第一層沒有 .pth／.pt／.ckpt／.safetensors 權重，請加入檔案後重跑。")

REPO = "Minervamuses/test"
COMMIT = "7a7c6ee8859754d5162bec032133b5bd7d2a103e"
SOURCE_ROOT = WORKSPACE / f"source-{COMMIT}"
package = SOURCE_ROOT / "drone_sr"
package.mkdir(parents=True, exist_ok=True)
for name in ("__init__.py", "__main__.py", "image_io.py", "inference.py", "tiling.py"):
    url = f"https://raw.githubusercontent.com/{REPO}/{COMMIT}/src/drone_sr/{name}"
    with urlopen(url, timeout=60) as response:
        data = response.read()
    compile(data, name, "exec")
    (package / name).write_bytes(data)

model_dropdown = widgets.Dropdown(
    options=[("請選擇權重", None)] + [(p.name, str(p)) for p in checkpoints],
    value=None,
    description="模型權重：",
    style={"description_width": "initial"},
    layout=widgets.Layout(width="100%"),
)
print("Source:", f"https://github.com/{REPO}/tree/{COMMIT}")
print(f"找到 {len(checkpoints)} 顆權重；請在下拉選單選擇後繼續。")
display(model_dropdown)

## 3. 選擇輸入方式

`upload` 模式每次只上傳一張圖片。`drive` 模式需把圖片先放入 Drive，路徑可從 Colab 左側檔案面板複製，例如 `/content/drive/MyDrive/drone_sr/input`；這裡填的是掛載後的路徑，不是分享網址。

Drive 原圖只讀取；`DRIVE_OUTPUT` 必須在輸入資料夾之外。`DRIVE_LIMIT=1` 先取依檔名排序的第一張，`0` 表示全部。修改設定後重新執行第 3–6 步。重跑第 4 步也會建立新的輸出目錄。

In [ ]:
#@title 輸入與輸出設定
INPUT_MODE = "upload" #@param ["upload", "drive"]
DRIVE_INPUT = "/content/drive/MyDrive/drone_sr/input" #@param {type:"string"}
DRIVE_OUTPUT = "/content/drive/MyDrive/drone_sr/output" #@param {type:"string"}
DRIVE_LIMIT = 1 #@param {type:"integer"}

input_directory = None
output_base = None
selected_images = []
last_output_directory = None
extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

if INPUT_MODE == "upload":
    upload_directory = Path(tempfile.mkdtemp(prefix="upload-", dir=WORKSPACE))
    uploaded = files.upload(target_dir=str(upload_directory))
    upload_count = len(uploaded)
    del uploaded
    if upload_count != 1:
        raise ValueError("請只選一張圖片；取消或多選時請重跑此步。")
    source_directory = upload_directory
    chosen_output_base = WORKSPACE / "results"
elif INPUT_MODE == "drive":
    if not isinstance(DRIVE_LIMIT, int) or DRIVE_LIMIT < 0:
        raise ValueError("DRIVE_LIMIT 必須為非負整數；0 代表全部。")
    drive.mount("/content/drive")
    drive_root = Path("/content/drive").resolve()
    source_directory = Path(DRIVE_INPUT).expanduser().resolve()
    chosen_output_base = Path(DRIVE_OUTPUT).expanduser().resolve()
    if not source_directory.is_relative_to(drive_root) or not chosen_output_base.is_relative_to(drive_root):
        raise ValueError("Drive 輸入／輸出必須使用 /content/drive/ 下的絕對路徑。")
    if chosen_output_base.is_relative_to(source_directory):
        raise ValueError("輸出資料夾不得等於或位於輸入資料夾內。")
else:
    raise ValueError("INPUT_MODE 必須是 upload 或 drive。")

if not source_directory.is_dir():
    raise FileNotFoundError(f"輸入資料夾不存在：{source_directory}")
candidates = sorted(
    (p for p in source_directory.iterdir() if p.is_file() and p.suffix.lower() in extensions),
    key=lambda p: (p.name.casefold(), p.name),
)
if not candidates:
    raise ValueError("資料夾第一層沒有支援的圖片（JPG/JPEG/PNG/TIF/TIFF）。")
selected_images = candidates[:DRIVE_LIMIT] if INPUT_MODE == "drive" and DRIVE_LIMIT else candidates
# Local symlinks select the requested images without copying or changing Drive originals.
staging = Path(tempfile.mkdtemp(prefix="input-", dir=WORKSPACE))
for source in selected_images:
    (staging / source.name).symlink_to(source.resolve())
chosen_output_base.mkdir(parents=True, exist_ok=True)
input_directory = staging
output_base = chosen_output_base
print(f"找到 {len(candidates)} 張，本次選取 {len(selected_images)} 張。")
print("輸出根目錄：", output_base)
for source in selected_images[:10]:
    print(" -", source.name)
if len(selected_images) > 10:
    print(f" ... 另 {len(selected_images) - 10} 張")

## 4. 執行超解析度

本步開始時讀取第 2 步下拉選單的當前選項，顯示權重路徑與 SHA-256，並寫入 `run.log`。若尚未選擇或檔案已移除，會停止並提示重新選擇。

逐張呼叫既有 CLI，一張失敗後仍處理下一張。超過 512 像素的圖片沿用 512 分塊／32 邊界，完整輸出在 RAM 組裝；分塊不代表任意大圖都能放進記憶體。先以一張觀察耗時及記憶體，再決定批次數量。

每輪產生新 `run-*` 資料夾，包含 PNG 與 `run.log`。退出碼非 0 代表有錯誤；已成功的結果仍可於後續步驟預覽、下載。

In [ ]:
last_output_directory = None
last_returncode = None
if input_directory is None or output_base is None:
    raise RuntimeError("請先成功完成第 3 步。")
MODEL_PATH = None
MODEL_SHA256 = None
if globals().get("model_dropdown") is None or model_dropdown.value is None:
    raise RuntimeError("請先執行第 2 步，並在下拉選單選擇模型權重。")
MODEL_PATH = Path(model_dropdown.value)
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f"權重不存在：{MODEL_PATH}；請重跑第 2 步更新清單。")
subprocess.run([sys.executable, "-c", GPU_CHECK], check=True)
with MODEL_PATH.open("rb") as stream:
    MODEL_SHA256 = hashlib.file_digest(stream, "sha256").hexdigest()
print("本次模型：", MODEL_PATH)
print("SHA-256：", MODEL_SHA256)
output_directory = Path(tempfile.mkdtemp(prefix="run-", dir=output_base))
env = os.environ.copy()
env["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + env.get("PYTHONPATH", "")
command = [
    sys.executable, "-u", "-m", "drone_sr", "--model", str(MODEL_PATH),
    "--input", str(input_directory), "--output", str(output_directory),
]
last_output_directory = output_directory
started = time.monotonic()
with (output_directory / "run.log").open("w", encoding="utf-8") as log:
    log.write(f"Source: {REPO}@{COMMIT}\nCheckpoint: {MODEL_PATH}\nCheckpoint SHA-256: {MODEL_SHA256}\n")
    log.write("Packages: " + json.dumps({n: version(n) for n in ("torch", "torchvision", "spandrel", "Pillow")}) + "\n")
    for source in selected_images:
        log.write(f"Input: {source} -> {source.stem}.png\n")
    log.flush()
    with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
        last_returncode = process.wait()
    elapsed = time.monotonic() - started
    log.write(f"Exit code: {last_returncode}\nElapsed seconds: {elapsed:.2f}\n")
print(f"退出碼：{last_returncode}；耗時：{elapsed:.2f} 秒")
print("本次結果：", output_directory)
if last_returncode != 0:
    print("本輪有失敗，請查看上述訊息與 run.log；下一步仍可檢視成功結果。")

## 5. 預覽第一張成功結果

顯示原圖與 SR 的縮圖，原尺寸 PNG 保留在輸出資料夾。這是視覺檢查，不是畫質提升的證明；沒有配對、對齊的真實高解析度參考圖，因此此處不計算 PSNR／SSIM，也不把生成紋理視為真實地物細節。

In [ ]:
from PIL import Image, ImageOps
from IPython.display import display

if last_output_directory is None:
    raise RuntimeError("請先執行第 4 步。")
results = sorted(last_output_directory.glob("*.png"))
print(f"本次產生 {len(results)} 個 PNG，退出碼：{last_returncode}")
if not results:
    print("沒有成功的圖片，請查看 run.log。")
else:
    result = results[0]
    original = next(p for p in selected_images if p.stem == result.stem)
    for label, path in (("Input", original), ("SR", result)):
        with Image.open(path) as image:
            preview = ImageOps.exif_transpose(image).convert("RGB")
            print(f"{label}: {path.name} — {preview.width} × {preview.height}")
            preview.thumbnail((768, 768))
            display(preview)

## 6. 下載結果

單張上傳模式會將本輪結果（含 `run.log`）打包下載。Drive 模式的結果已存於第 4 步列出的資料夾，預設不再複製壓縮；需要下載到電腦時勾選 `DOWNLOAD_DRIVE_ZIP`。大量 PNG 打包需要額外暫存空間及時間。

In [ ]:
#@title 儲存結果
DOWNLOAD_DRIVE_ZIP = False #@param {type:"boolean"}

if last_output_directory is None:
    raise RuntimeError("尚無執行結果，請先執行第 4 步。")
print("結果與 log：", last_output_directory)
if INPUT_MODE == "upload" or DOWNLOAD_DRIVE_ZIP:
    archive_directory = Path(tempfile.mkdtemp(prefix="download-", dir=WORKSPACE))
    archive = shutil.make_archive(str(archive_directory / "drone-sr-results"), "zip", last_output_directory)
    files.download(archive)
else:
    print("結果已寫入 Google Drive；請在 Drive 確認檔案已同步。")